# Road PCI Intelligence — Step 3B: Road PCI Intelligence - Train Masked (Run A)

**Experiment Configuration**:
- **Run Identifier**: `run_a_masked`
- **Loss Formulation**: `Masked` Multi-Task Loss (`bce+dice`)
  - In `masked` mode: channels are supervised strictly over samples labeling that distress.
  - In `naive` mode: all channels supervised across all samples, treating unlabelled channels as all-zero negatives.
- **Model Architecture**: U-Net with ResNet-34 encoder pretrained on ImageNet (`segmentation-models-pytorch`).
- **Epochs**: `30`
- **Random Seed**: `42`
- **Optimization**: AdamW (`lr=3e-4`, `weight_decay=1e-4`), Cosine Annealing, AMP.
- **Evaluation**: Full native resolution validation (batch size 1, unpadded back to original dimensions).
- **Selection Metric**: Mean(CRACK500 Dataset Dice, Pothole Dataset Dice) on validation split.


In [ ]:
# ==============================================================================
# 1. SETUP ENVIRONMENT, DEPENDENCIES & REPO REPLICATION
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

# 1. Install required packages
print("Installing segmentation-models-pytorch and albumentations...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8"], check=True)

# 2. Clone/pull GitHub repository for clean modular code and manifests
repo_dir = Path('/kaggle/working/repo')
if not repo_dir.exists():
    print("Cloning GitHub repository...")
    subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
else:
    print("Pulling latest GitHub changes...")
    subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import torch
import albumentations as A
import segmentation_models_pytorch as smp

from src.train import seed_everything, log_library_versions

seed_everything(42)
log_library_versions()

assert torch.cuda.is_available(), "CRITICAL: CUDA is not available! Enable GPU accelerator in Kaggle notebook settings."
print(f"CUDA Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==============================================================================
# 2. RESOLVE MANIFESTS & DATASET ROOTS
# ==============================================================================
from pathlib import Path
import pandas as pd

from src.build_manifest import DATASET_SPECS, resolve_dataset_root

manifest_path = repo_dir / 'manifests/segmentation_manifest.csv'
assert manifest_path.exists(), f"Missing manifest: {manifest_path}"
manifest_df = pd.read_csv(manifest_path)
print(f"Loaded segmentation manifest: {len(manifest_df):,} records")

# Resolve dataset roots
dataset_roots = {}
for src in ['pothole', 'road_crack', 'crack500']:
    spec = DATASET_SPECS[src]
    root = resolve_dataset_root(spec['owner'], spec['name'])
    assert root is not None and root.exists(), f"Could not find dataset root for {src}"
    dataset_roots[src] = root
    print(f"  • {src:10s} root: {root}")


In [ ]:
# ==============================================================================
# 3. RUN FULL BASELINE TRAINING (30 EPOCHS, MASKED LOSS)
# ==============================================================================
import torch
from pathlib import Path
from src.train import run_full_training

output_dir = Path('/kaggle/working/run_a_masked')
output_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model, history, sweep_results = run_full_training(
    manifest_df=manifest_df,
    dataset_roots=dataset_roots,
    epochs=30,
    loss_mode='masked',
    loss_terms='bce+dice',
    seed=42,
    output_dir=output_dir,
    lr=3e-4,
    weight_decay=1e-4,
    batch_size=8,
    samples_per_epoch=600,
    device=device
)

print("\n[PASS] Training and validation threshold sweep completed successfully.")


In [ ]:
# ==============================================================================
# 4. DISPLAY LEARNING CURVES & SUMMARY
# ==============================================================================
import matplotlib.pyplot as plt
from PIL import Image

lc_path = output_dir / 'learning_curves.png'
if lc_path.exists():
    img = Image.open(lc_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Road PCI Intelligence - Train Masked (Run A) — Learning Curves", fontsize=14, weight='bold', pad=10)
    plt.show()

print("\nArtifacts saved in", output_dir)
for p in sorted(output_dir.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  • {p.name:<25} ({size_kb:.1f} KB)")
